Trong phần này sẽ có tiền xử lý + smote + gan + train test

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [ ]:
data.duplicated().sum()


np.int64(13)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.isna().sum().sum()

np.int64(0)

In [ ]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
baseline value,2113.0,133.304780,9.837451,106.0,126.000,133.000,140.000,160.000
accelerations,2113.0,0.003188,0.003871,0.0,0.000,0.002,0.006,0.019
fetal_movement,2113.0,0.009517,0.046804,0.0,0.000,0.000,0.003,0.481
uterine_contractions,2113.0,0.004387,0.002941,0.0,0.002,0.005,0.007,0.015
light_decelerations,2113.0,0.001901,0.002966,0.0,0.000,0.000,0.003,0.015
severe_decelerations,2113.0,0.000003,0.000057,0.0,0.000,0.000,0.000,0.001
prolongued_decelerations,2113.0,0.000159,0.000592,0.0,0.000,0.000,0.000,0.005
abnormal_short_term_variability,2113.0,46.993848,17.177782,12.0,32.000,49.000,61.000,87.000
mean_value_of_short_term_variability,2113.0,1.335021,0.884368,0.2,0.700,1.200,1.700,7.000
percentage_of_time_with_abnormal_long_term_variability,2113.0,9.795078,18.337073,0.0,0.000,0.000,11.000,91.000


In [ ]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [ ]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1110
1.0,838
-1.0,165


In [ ]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1646
2.0,292
3.0,175


GAN

In [ ]:
!pip install ctgan

GAN

In [ ]:
import pandas as pd
from ctgan import CTGAN
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

target_col = 'fetal_health'

print("Kích thước data gốc:", data.shape)
display(data.head())

print("\nPhân bố lớp ban đầu:")
print(data[target_col].value_counts().sort_index())

# =============================
# 1. Train CTGAN trên toàn bộ data
# =============================
gan_df = data.copy()

ctgan = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan.fit(gan_df, discrete_columns=[target_col])

print("\nĐã train xong CTGAN")

# =============================
# 2. GAN chỉ sinh MỘT PHẦN để gần cân bằng
# =============================
class_counts = gan_df[target_col].value_counts().sort_index()
max_count = class_counts.max()

# GAN chỉ bù một phần khoảng thiếu
gan_fill_ratio = 0.6   # có thể đổi 0.5, 0.6, 0.7 tùy bạn

synthetic_parts = []

for cls, count in class_counts.items():
    gap = max_count - count
    need = int(gap * gan_fill_ratio)

    if need <= 0:
        continue

    print(f"\nLớp {cls}: thiếu {gap} mẫu, GAN sẽ sinh thêm {need} mẫu")

    collected = []
    total_collected = 0

    while total_collected < need:
        sample_n = max(500, need * 3)
        fake_batch = ctgan.sample(sample_n)

        fake_cls = fake_batch[fake_batch[target_col] == cls].copy()

        if len(fake_cls) > 0:
            collected.append(fake_cls)
            total_collected += len(fake_cls)
            print(f"Đã lấy được {total_collected}/{need}")

    fake_cls_final = pd.concat(collected, axis=0).iloc[:need].copy()
    synthetic_parts.append(fake_cls_final)

# =============================
# 3. Gộp dữ liệu thật + dữ liệu GAN một phần
# =============================
if len(synthetic_parts) > 0:
    synthetic_df = pd.concat(synthetic_parts, axis=0).reset_index(drop=True)
    semi_balanced_df = pd.concat([gan_df, synthetic_df], axis=0).reset_index(drop=True)
else:
    semi_balanced_df = gan_df.copy()

print("\nPhân bố lớp sau GAN một phần:")
print(semi_balanced_df[target_col].value_counts().sort_index())

# =============================
# 4. Tách X, y
# =============================
X = semi_balanced_df.drop(columns=[target_col]).copy()
y = semi_balanced_df[target_col].astype(int)

# =============================
# 5. Dùng SMOTE để cân bằng hoàn toàn
# =============================
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

X_smote = pd.DataFrame(X_smote, columns=X.columns)
y_smote = pd.Series(y_smote, name=target_col)

balanced_df = pd.concat([X_smote, y_smote], axis=1)

print("\nPhân bố lớp sau GAN + SMOTE:")
print(balanced_df[target_col].value_counts().sort_index())

# =============================
# 6. Chia train/test
# =============================
X_final = balanced_df.drop(columns=[target_col])
y_final = balanced_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.2,
    random_state=42,
    stratify=y_final
)

print("\nKích thước tập train/test:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nPhân bố y_train:")
print(y_train.value_counts().sort_index())

print("\nPhân bố y_test:")
print(y_test.value_counts().sort_index())

Kích thước data gốc: (2113, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0



Phân bố lớp ban đầu:
fetal_health
1.0    1646
2.0     292
3.0     175
Name: count, dtype: int64


Gen. (-02.24) | Discrim. (+00.64): 100%|██████████| 300/300 [05:58<00:00,  1.20s/it]



Đã train xong CTGAN

Lớp 2.0: thiếu 1354 mẫu, GAN sẽ sinh thêm 812 mẫu
Đã lấy được 760/812
Đã lấy được 1524/812

Lớp 3.0: thiếu 1471 mẫu, GAN sẽ sinh thêm 882 mẫu
Đã lấy được 761/882
Đã lấy được 1561/882

Phân bố lớp sau GAN một phần:
fetal_health
1.0    1646
2.0    1104
3.0    1057
Name: count, dtype: int64

Phân bố lớp sau GAN + SMOTE:
fetal_health
1    1646
2    1646
3    1646
Name: count, dtype: int64

Kích thước tập train/test:
X_train: (3950, 21)
X_test : (988, 21)

Phân bố y_train:
fetal_health
1    1317
2    1317
3    1316
Name: count, dtype: int64

Phân bố y_test:
fetal_health
1    329
2    329
3    330
Name: count, dtype: int64


In [ ]:
# =============================
# 5. Scale sau khi chia
# =============================
scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("\nĐã scale xong dữ liệu")

# =============================
# 6. Train model
# =============================
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

# =============================
# 7. Đánh giá
# =============================
print("\n=== KẾT QUẢ ===")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nKQ FOR EXCEL\n")
print("Precision weighted:", precision_score(y_test, y_pred, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred, average='macro'))
print("\nDetail about one class:")


print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nConfusion Matrix (normalize='true'):")
print(confusion_matrix(y_test, y_pred, normalize='true'))


Đã scale xong dữ liệu

=== KẾT QUẢ ===
Accuracy: 0.9595141700404858

KQ FOR EXCEL

Precision weighted: 0.9594081660080706
Recall weighted: 0.9595141700404858
F1 weighted: 0.959419761509324
Precision macro: 0.9594158670588228
Recall macro: 0.9595161339842191
F1 macro: 0.9594245608832379

Detail about one class:
Balanced Accuracy: 0.9595161339842191
Macro F1: 0.9594245608832379

Classification Report:
              precision    recall  f1-score   support

           1     0.9760    0.9878    0.9819       329
           2     0.9505    0.9331    0.9417       329
           3     0.9518    0.9576    0.9547       330

    accuracy                         0.9595       988
   macro avg     0.9594    0.9595    0.9594       988
weighted avg     0.9594    0.9595    0.9594       988


Confusion Matrix:
[[325   4   0]
 [  6 307  16]
 [  2  12 316]]

Confusion Matrix (normalize='true'):
[[0.98784195 0.01215805 0.        ]
 [0.01823708 0.9331307  0.04863222]
 [0.00606061 0.03636364 0.95757576]]
